## 1. Load, Split Data

In [19]:
import kagglehub
from kagglehub import KaggleDatasetAdapter

# https://www.kaggle.com/datasets/uciml/sms-spam-collection-dataset?select=spam.csv
df = kagglehub.load_dataset(
    KaggleDatasetAdapter.PANDAS,
    'uciml/sms-spam-collection-dataset',
    'spam.csv',
    pandas_kwargs={"encoding": "latin-1"}
)

# Check data
df.head()

/var/folders/hg/xkc2k9pn7s94gjhll09zr_bw0000gn/T/ipykernel_74706/591930350.py:5: DeprecationWarning: Use dataset_load() instead of load_dataset(). load_dataset() will be removed in a future version.
  df = kagglehub.load_dataset(


,v1,v2,Unnamed: 2,Unnamed: 3,Unnamed: 4
0,ham,"Go until jurong point, crazy.. Available only ...",NaN,NaN,NaN
1,ham,Ok lar... Joking wif u oni...,NaN,NaN,NaN
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,NaN,NaN,NaN
3,ham,U dun say so early hor... U c already then say...,NaN,NaN,NaN
4,ham,"Nah I don't think he goes to usf, he lives aro...",NaN,NaN,NaN


In [20]:
# Check spam / ham ratio
df['v1'].value_counts()

v1
ham     4825
spam     747
Name: count, dtype: int64

In [21]:
from sklearn.model_selection import train_test_split

# Assign feats, targets
X = df['v2']
y = df['v1']

# Split train, test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, train_size=0.3, stratify=y
)

# Check ratio after split
print(y_train.value_counts())
print(y_test.value_counts())

v1
ham     1447
spam     224
Name: count, dtype: int64
v1
ham     3378
spam     523
Name: count, dtype: int64


---

## 2. Encoding

In [22]:
from sklearn.preprocessing import LabelEncoder

# Transform spam, ham to 0, 1
le = LabelEncoder()
y_train = le.fit_transform(y_train)
y_test = le.transform(y_test)

# See which index is ham
for i, name in enumerate(le.classes_):
    print(i, "=", name)

0 = ham
1 = spam


In [23]:
from sklearn.pipeline import Pipeline
from sklearn.dummy import DummyClassifier

# Init empty pipe
pipe = Pipeline([
    ('vec', None),
    ('model', DummyClassifier)
])

---

## 3. Vectorizers (Count, TF-IDF)

In [24]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

# Count vectorizer
count_params = {
    'vec': [CountVectorizer()],
    'vec__ngram_range': [(1, 1), (1, 2), (1, 3), (2, 3)],
    'vec__min_df': [0.001, 0.01],
    'vec__max_df': [0.9, 0.95],
    'vec__stop_words': ['english'],
    'vec__binary': [True]
}

# TF-IDF vectorizer
tfidf_params = {
    **count_params,
    'vec': [TfidfVectorizer()],     # Overwrite count vectorizer
    'vec__sublinear_tf': [True, False],
    'vec__norm': [None, 'l2']
}

---

## 4. Models (LogisticRegression, BernoulliNB, MLPClassifier)

In [25]:
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import BernoulliNB
from sklearn.neural_network import MLPClassifier

# Logistic regression
logistic_params = {
    'model': [
        LogisticRegression(
            max_iter=1000,
            class_weight='balanced',
            random_state=42
        )
    ],
    'model__C': [0.01, 0.1, 1, 10],
    'model__solver': ['lbfgs', 'saga']
}

# Bernoulli NB
bernoulli_params = {
    'model': [
        BernoulliNB()
    ],
    'model__alpha': [0.001, 0.01, 0.1]
}

# MLP
mlp_params = {
    'model': [
        MLPClassifier(
            activation='relu',
            solver='adam',
            random_state=42
        )
    ],
    'model__hidden_layer_sizes': [(20, 20), (20, 30, 20)],
    'model__learning_rate_init': [0.001, 0.01, 0.1],
    'model__alpha': [0.001, 0.01, 0.1],
    'model__max_iter': [3000, 4000]
}

---

## 5. Test Different Combinations

In [26]:
vec_model_params = []

# Generate all permutation
for vec_params in [count_params, tfidf_params]:
    for model_params in [logistic_params, bernoulli_params, mlp_params]:
        vec_model_params.append(vec_params | model_params)
vec_model_params[0:2]

[{'vec': [CountVectorizer()],
  'vec__ngram_range': [(1, 1), (1, 2), (1, 3), (2, 3)],
  'vec__min_df': [0.001, 0.01],
  'vec__max_df': [0.9, 0.95],
  'vec__stop_words': ['english'],
  'vec__binary': [True],
  'model': [LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)],
  'model__C': [0.01, 0.1, 1, 10],
  'model__solver': ['lbfgs', 'saga']},
 {'vec': [CountVectorizer()],
  'vec__ngram_range': [(1, 1), (1, 2), (1, 3), (2, 3)],
  'vec__min_df': [0.001, 0.01],
  'vec__max_df': [0.9, 0.95],
  'vec__stop_words': ['english'],
  'vec__binary': [True],
  'model': [BernoulliNB()],
  'model__alpha': [0.001, 0.01, 0.1]}]

In [27]:
from sklearn.pipeline import Pipeline
from sklearn.dummy import DummyClassifier
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold

best = None

# Init empty pipe
pipe = Pipeline([
    ('vec', None),
    ('model', DummyClassifier)
])

cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

# Iterate combinations
for params in vec_model_params:
    search = RandomizedSearchCV(
        estimator=pipe,
        param_distributions=params,
        cv=cv,
        scoring='f1',
        n_jobs=1,
        n_iter=10,
        random_state=13,
        error_score='raise'
    )
    search.fit(X_train, y_train)

    # Keep track of best
    if best is None or search.best_score_ > best.best_score_:
        best = search

---

## 6. Results

In [28]:
print(f'Best CV F1: {best.best_score_:.4f}')
print(f'Best estimator:\n{best.best_estimator_}')
print(f'Best params:\n{best.best_params_}') 

Best CV F1: 0.9401
Best estimator:
Pipeline(steps=[('vec',
                 TfidfVectorizer(binary=True, max_df=0.95, min_df=0.001,
                                 stop_words='english', sublinear_tf=True)),
                ('model',
                 LogisticRegression(C=10, class_weight='balanced',
                                    max_iter=1000, random_state=42,
                                    solver='saga'))])
Best params:
{'vec__sublinear_tf': True, 'vec__stop_words': 'english', 'vec__norm': 'l2', 'vec__ngram_range': (1, 1), 'vec__min_df': 0.001, 'vec__max_df': 0.95, 'vec__binary': True, 'vec': TfidfVectorizer(), 'model__solver': 'saga', 'model__C': 10, 'model': LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)}


In [30]:
should_ham = search.predict(['Go until jurong point, crazy.. Available only in bugis n great world la e buffet... Cine there got amore wat...'])
should_spam = search.predict(['SIX chances to win CASH! From 100 to 20,000 pounds txt> CSH11 and send to 87575. Cost 150p/day, 6days, 16+ TsandCs apply Reply HL 4 info'])

print(f'Should ham: {should_ham}')
print(f'Should spam: {should_spam}')

Should ham: [0]
Should spam: [1]
